# Patent Agent MVP Pipeline (PatentFlowAI)
User Query\
    ↓\
PatentRootAgent-LLM (delegates)\
    ↓\
SequentialResearchPipeline\
    ↓\
ParallelSearchAgent (concurrent execution)\
    ├─> PatentsSearchAgent-LLM ✓ (A Plug with 3 preloaded patents)\
    └─> GoogleSearchAgent-LLM ✓ (found research gaps)\
    ↓\
DataCleaningAgent-LLM ✓ (implicit - no explicit call shown, but data was processed)\
    ↓\
AnalysisAgent-LLM ✓ (generated complete report)\
    ↓\
    PatentRootAgent-LLM (summarize report)\
    ↓\
Success!

In [1]:
# ==============================================================================
# Patent Research Assistant MVP - Enhanced Observability Version
# ==============================================================================
# This version incorporates production-grade logging, metrics tracking,
# error handling, and validation while maintaining the original architecture.
# ==============================================================================

import os
import logging
import sys
import warnings
import time
import json
from datetime import datetime
from typing import Union, Dict, Any, Optional
from dataclasses import dataclass, asdict

# Google ADK imports
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent
from google.adk.agents.base_agent import BaseAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.google_llm import Gemini
from google.adk.models.llm_request import LlmRequest
from google.adk.tools import FunctionTool, google_search
from google.adk.runners import InMemoryRunner
from google.adk.plugins.logging_plugin import LoggingPlugin
from google.adk.plugins.base_plugin import BasePlugin
from google.genai import types
from kaggle_secrets import UserSecretsClient

In [2]:
# ==============================================================================
# SECTION 1: Logging and Observability Configuration
# ==============================================================================

# Create logs directory
LOGS_DIR = "/kaggle/working/logs"
os.makedirs(LOGS_DIR, exist_ok=True)

# Configure comprehensive logging
LOG_FILENAME = f"{LOGS_DIR}/patent_research_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)-25s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_FILENAME, mode='w', encoding='utf-8')
    ],
    force=True  # Override any existing configuration
)

# Set specific log levels for libraries
logging.getLogger('google.genai').setLevel(logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('google.adk').setLevel(logging.INFO)

# Suppress noisy warnings
warnings.filterwarnings('ignore', message='.*non-text parts in the response.*')
warnings.filterwarnings('ignore', module='google_genai.types')

# Create module logger
logger = logging.getLogger(__name__)

# ==============================================================================
# SECTION 2: Metrics and Observability Classes
# ==============================================================================

@dataclass
class ToolMetrics:
    """Metrics for individual tool executions"""
    tool_name: str
    agent_name: str
    start_time: float
    end_time: float
    duration_seconds: float
    status: str  # 'success' or 'error'
    result_size: int = 0
    error_message: Optional[str] = None

@dataclass
class AgentMetrics:
    """Metrics for agent executions"""
    agent_name: str
    start_time: float
    end_time: Optional[float] = None
    duration_seconds: Optional[float] = None
    status: str = 'in_progress'  # 'in_progress', 'success', 'error'
    tool_calls: int = 0

class MetricsCollector:
    """Collects and tracks execution metrics"""
    
    def __init__(self):
        self.pipeline_start_time = None
        self.pipeline_end_time = None
        self.tool_metrics = []
        self.agent_metrics = []
        self.events = []
        self.logger = logging.getLogger("MetricsCollector")
        
    def start_pipeline(self):
        """Mark pipeline start"""
        self.pipeline_start_time = time.time()
        self.logger.info("=" * 70)
        self.logger.info("🚀 PIPELINE EXECUTION STARTED")
        self.logger.info("=" * 70)
    
    def end_pipeline(self, success: bool = True):
        """Mark pipeline end and log summary"""
        self.pipeline_end_time = time.time()
        duration = self.pipeline_end_time - self.pipeline_start_time
        
        status = "✅ SUCCESS" if success else "❌ FAILED"
        self.logger.info("=" * 70)
        self.logger.info(f"{status} - PIPELINE EXECUTION COMPLETED")
        self.logger.info(f"Total Duration: {duration:.2f} seconds")
        self.logger.info(f"Total Tool Calls: {len(self.tool_metrics)}")
        self.logger.info(f"Agents Tracked: {len([m for m in self.agent_metrics if m.end_time is not None])}")
        self.logger.info("=" * 70)
    
    def log_tool_execution(self, metrics: ToolMetrics):
        """Log tool execution metrics"""
        self.tool_metrics.append(metrics)
        
        status_icon = "✅" if metrics.status == "success" else "❌"
        self.logger.info(
            f"{status_icon} Tool: {metrics.tool_name} | "
            f"Agent: {metrics.agent_name} | "
            f"Duration: {metrics.duration_seconds:.2f}s | "
            f"Result Size: {metrics.result_size} chars"
        )
        
        if metrics.error_message:
            self.logger.error(f"   Error: {metrics.error_message}")
    
    def log_agent_start(self, agent_name: str):
        """Log agent start"""
        agent_metrics = AgentMetrics(
            agent_name=agent_name,
            start_time=time.time()
        )
        self.agent_metrics.append(agent_metrics)
        self.logger.info(f"🤖 Agent Started: {agent_name}")
        return agent_metrics
    
    def log_agent_end(self, agent_name: str, success: bool = True):
        """Log agent completion"""
        for agent_metric in reversed(self.agent_metrics):
            if agent_metric.agent_name == agent_name and agent_metric.end_time is None:
                agent_metric.end_time = time.time()
                agent_metric.duration_seconds = agent_metric.end_time - agent_metric.start_time
                agent_metric.status = 'success' if success else 'error'
                
                status_icon = "✅" if success else "❌"
                self.logger.info(
                    f"{status_icon} Agent Completed: {agent_name} | "
                    f"Duration: {agent_metric.duration_seconds:.2f}s"
                )
                break
    
    def export_metrics(self, filepath: Optional[str] = None) -> str:
        """Export all metrics to JSON file"""
        if filepath is None:
            filepath = f"{LOGS_DIR}/metrics_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
        metrics_data = {
            "pipeline": {
                "start_time": datetime.fromtimestamp(self.pipeline_start_time).isoformat() if self.pipeline_start_time else None,
                "end_time": datetime.fromtimestamp(self.pipeline_end_time).isoformat() if self.pipeline_end_time else None,
                "duration_seconds": (self.pipeline_end_time - self.pipeline_start_time) if self.pipeline_end_time else None
            },
            "tools": [asdict(m) for m in self.tool_metrics],
            "agents": [asdict(m) for m in self.agent_metrics],
            "summary": {
                "total_tools": len(self.tool_metrics),
                "successful_tools": len([m for m in self.tool_metrics if m.status == "success"]),
                "total_agents_tracked": len(self.agent_metrics),
                "completed_agents": len([m for m in self.agent_metrics if m.end_time is not None])
            }
        }
        
        with open(filepath, 'w') as f:
            json.dump(metrics_data, f, indent=2)
        
        self.logger.info(f"📊 Metrics exported to: {filepath}")
        return filepath

# Initialize global metrics collector
metrics = MetricsCollector()

# ==============================================================================
# SECTION 2.1: Custom Observability Plugin
# ==============================================================================

class ObservabilityPlugin(BasePlugin):
    """
    Custom plugin that tracks agent invocations and integrates with MetricsCollector.
    Uses ADK's callback system to capture agent lifecycle events.
    """
    
    def __init__(self, metrics_collector: MetricsCollector) -> None:
        """Initialize the plugin with a metrics collector reference."""
        super().__init__(name="observability_plugin")
        self.metrics = metrics_collector
        self.agent_count = 0
        self.llm_request_count = 0
        self.logger = logging.getLogger("ObservabilityPlugin")
    
    async def before_agent_callback(
        self, *, agent: BaseAgent, callback_context: CallbackContext
    ) -> None:
        """
        Called before each agent execution.
        Tracks agent start time and logs invocation.
        """
        self.agent_count += 1
        agent_name = agent.name if hasattr(agent, 'name') else agent.__class__.__name__
        
        self.logger.info(f"[Plugin] Agent #{self.agent_count}: {agent_name} starting...")
        self.metrics.log_agent_start(agent_name)
    
    async def after_agent_callback(
        self, *, agent: BaseAgent, callback_context: CallbackContext
    ) -> None:
        """
        Called after each agent execution.
        Tracks agent completion and duration.
        """
        agent_name = agent.name if hasattr(agent, 'name') else agent.__class__.__name__
        self.logger.info(f"[Plugin] Agent completed: {agent_name}")
        self.metrics.log_agent_end(agent_name, success=True)
    
    async def before_model_callback(
        self, *, callback_context: CallbackContext, llm_request: LlmRequest
    ) -> None:
        """
        Called before each LLM API call.
        Tracks model invocations for cost/performance analysis.
        """
        self.llm_request_count += 1
        self.logger.debug(f"[Plugin] LLM request #{self.llm_request_count}")
    
    def get_summary(self) -> Dict[str, int]:
        """Return summary statistics"""
        return {
            "total_agent_invocations": self.agent_count,
            "total_llm_requests": self.llm_request_count,
            "completed_agents": len([m for m in self.metrics.agent_metrics if m.end_time is not None])
        }

# ==============================================================================
# SECTION 3: Configuration
# ==============================================================================

logger.info("Initializing Patent Research Assistant MVP")
logger.info(f"Log file: {LOG_FILENAME}")

# API Configuration
user_secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

logger.info("✅ API credentials configured")

# Retry configuration for resilience
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

MODEL_NAME = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config)
APP_NAME = "patent_research_assistant"

logger.info(f"✅ Model configured: gemini-2.5-flash-lite with retry logic")

# ==============================================================================
# SECTION 4: Instrumented Tools with Observability
# ==============================================================================

def google_patents_api_tool(query: str) -> dict:
    """
    Retrieves structured patent data from Google Patents database.
    Instrumented with logging and metrics.
    
    Args:
        query: The specific invention or patent class to search for
        
    Returns:
        Dictionary containing patent results with metadata
    """
    tool_logger = logging.getLogger("PatentsTool")
    start_time = time.time()
    
    tool_logger.info(f"🔍 Searching patents for: '{query[:60]}...'")
    
    try:
        # Validation
        if not query or len(query.strip()) < 3:
            tool_logger.warning(f"⚠️  Query too short: '{query}'")
            result = {"status": "error", "message": "Query too short"}
            result_size = len(str(result))
        else:
            # Generic mock data that works for ANY query
            result = {
                "status": "success",
                "query": query,
                "result_count": 3,
                "results": [
                    {
                        "id": "US20210012345A1",
                        "title": f"Advanced System for {query[:30]}",
                        "class": "F16K1/00",
                        "abstract": f"Innovation related to {query} with improved performance characteristics...",
                        "year": 2021
                    },
                    {
                        "id": "US20220067890A1",
                        "title": f"Novel Method for {query[:30]}",
                        "class": "F16K31/00",
                        "abstract": f"Process improvements for {query} in industrial applications...",
                        "year": 2022
                    },
                    {
                        "id": "US20230024680A1",
                        "title": f"Enhanced Design for {query[:30]}",
                        "class": "F16K47/00",
                        "abstract": f"Next-generation approach to {query} with cost optimization...",
                        "year": 2023
                    }
                ]
            }
            result_size = len(json.dumps(result))
            tool_logger.info(f"✅ Found {result['result_count']} patents for query: '{query[:40]}...'")
        
        # Log metrics
        duration = time.time() - start_time
        metrics.log_tool_execution(ToolMetrics(
            tool_name="google_patents_api_tool",
            agent_name="PatentsSearchAgent",
            start_time=start_time,
            end_time=time.time(),
            duration_seconds=duration,
            status="success",
            result_size=result_size
        ))
        
        return result
        
    except Exception as e:
        duration = time.time() - start_time
        tool_logger.error(f"❌ Patent search failed: {e}", exc_info=True)
        
        metrics.log_tool_execution(ToolMetrics(
            tool_name="google_patents_api_tool",
            agent_name="PatentsSearchAgent",
            start_time=start_time,
            end_time=time.time(),
            duration_seconds=duration,
            status="error",
            error_message=str(e)
        ))
        
        return {"status": "error", "message": str(e)}

def data_cleaner_tool(raw_data: Any) -> str:
    """
    Cleans and normalizes data from search tools.
    Instrumented with logging and metrics.
    
    Args:
        raw_data: Output from previous search tools
        
    Returns:
        Cleaned and formatted text ready for analysis
    """
    tool_logger = logging.getLogger("CleanerTool")
    start_time = time.time()
    
    tool_logger.info(f"🧹 Cleaning data (type: {type(raw_data).__name__})")
    
    try:
        # Convert any data type to string
        if isinstance(raw_data, (dict, list)):
            content_str = json.dumps(raw_data, indent=2, ensure_ascii=False)
        else:
            content_str = str(raw_data)
        
        clean_text = content_str.strip()
        
        final_output = (
            f"--- BEGIN CLEANED DATA ---\n"
            f"{clean_text}\n"
            f"--- END CLEANED DATA ---\n"
            f"System Note: Data normalized and ready for analysis."
        )
        
        duration = time.time() - start_time
        tool_logger.info(f"✅ Cleaned {len(final_output)} characters in {duration:.2f}s")
        
        metrics.log_tool_execution(ToolMetrics(
            tool_name="data_cleaner_tool",
            agent_name="DataCleaningAgent",
            start_time=start_time,
            end_time=time.time(),
            duration_seconds=duration,
            status="success",
            result_size=len(final_output)
        ))
        
        return final_output
        
    except Exception as e:
        duration = time.time() - start_time
        tool_logger.error(f"❌ Data cleaning failed: {e}", exc_info=True)
        
        metrics.log_tool_execution(ToolMetrics(
            tool_name="data_cleaner_tool",
            agent_name="DataCleaningAgent",
            start_time=start_time,
            end_time=time.time(),
            duration_seconds=duration,
            status="error",
            error_message=str(e)
        ))
        
        return f"ERROR: Failed to clean data - {str(e)}"

# Create tool instances
PATENTS_TOOL = FunctionTool(google_patents_api_tool)
CLEANER_TOOL = FunctionTool(data_cleaner_tool)
GOOGLE_SEARCH_TOOL = google_search

logger.info("✅ Tools initialized with instrumentation")

# ==============================================================================
# SECTION 5: Agent Configuration
# ==============================================================================

logger.info("Configuring agent hierarchy...")

PATENTS_SEARCH_AGENT = LlmAgent(
    name="PatentsSearchAgent",
    model=MODEL_NAME,
    instruction="""You are a patent search executor.

CRITICAL: You must call the function 'google_patents_api_tool' (exact name).

TASK:
1. Extract key search terms from the user's query
2. Call: google_patents_api_tool(query="your search terms")
3. Return the raw tool output immediately

FORBIDDEN:
- Do NOT call functions named 'search', 'patent_search', or any other name
- Do NOT analyze results
- Do NOT transfer control

The ONLY function available is 'google_patents_api_tool'.""",
    tools=[PATENTS_TOOL]
)

GOOGLE_SEARCH_AGENT = LlmAgent(
    name="GoogleSearchAgent",
    model=MODEL_NAME,
    instruction="""You are a specialized web search agent. Your ONLY job is:
    1. Use google_search to find relevant information
    2. Return CONCISE results (max 300 words)
    3. Focus on key findings only
    
    DO NOT:
    - Transfer control to other agents
    - Provide lengthy explanations
    - Ask follow-up questions
    
    Execute the search and return brief, focused results.""",
    tools=[GOOGLE_SEARCH_TOOL]
)

CLEANING_AGENT = LlmAgent(
    name="DataCleaningAgent",
    model=MODEL_NAME,
    instruction="""You receive raw search results from multiple agents.

YOUR TASK:
1. Call the data_cleaner_tool function with ALL raw input
2. Pass the complete content as the 'raw_data' parameter
3. Return ONLY the cleaned output from the tool

CRITICAL: The tool is named 'data_cleaner_tool' exactly.

DO NOT add commentary or explanations. Just clean and return.""",
    tools=[CLEANER_TOOL]
)

ANALYSIS_AGENT = LlmAgent(
    model=MODEL_NAME,
    name="AnalysisAgent",
    instruction="""You are the Patent Insight Specialist. Analyze the CLEANED data.

OUTPUT STRUCTURE:
1. **Technology Classes**: List CPC/IPC codes found
2. **Main Inventions**: Summarize 3 key patents
3. **White Spots**: Identify 3-5 unexplored R&D areas with detailed explanations

You do NOT use tools. Analyze only the provided context."""
)

# Orchestration layers
PARALLEL_SEARCH_AGENT = ParallelAgent(
    name="ParallelSearchAgent",
    sub_agents=[PATENTS_SEARCH_AGENT, GOOGLE_SEARCH_AGENT],
    description="Executes patent and web searches concurrently for speed."
)

SEQUENTIAL_RESEARCH_PIPELINE = SequentialAgent(
    name="SequentialResearchPipeline",
    description="Coordinates parallel research, cleaning, and analysis.",
    sub_agents=[
        PARALLEL_SEARCH_AGENT,
        CLEANING_AGENT,
        ANALYSIS_AGENT
    ]
)

PATENT_ROOT_AGENT = LlmAgent(
    model=MODEL_NAME,
    name="PatentRootAgent",
    instruction="""You are the Patent Research Coordinator. 
    Delegate the user's request to the Sequential Research Pipeline.
    Present the final analysis clearly to the user.""",
    sub_agents=[SEQUENTIAL_RESEARCH_PIPELINE]
)

logger.info("✅ Agent hierarchy configured")
logger.info("   - Root: PatentRootAgent")
logger.info("   - Pipeline: SequentialResearchPipeline")
logger.info("   - Parallel: PatentsSearchAgent + GoogleSearchAgent")
logger.info("   - Sequential: CleaningAgent → AnalysisAgent")

# ==============================================================================
# SECTION 6: Response Validation
# ==============================================================================

def validate_response(response) -> Dict[str, Any]:
    """
    Validate the quality of the agent's response.
    
    Returns:
        Validation results with pass/fail status and details
    """
    try:
        text = response.final_response.text if hasattr(response, 'final_response') else str(response)
        
        checks = {
            "has_technology_classes": {
                "passed": any(code in text for code in ["H01", "C01", "G01", "B01"]),
                "message": "Technology class codes found"
            },
            "has_white_spots": {
                "passed": "white spot" in text.lower() or "gap" in text.lower(),
                "message": "White spots identified"
            },
            "has_multiple_insights": {
                "passed": text.count("**") >= 6,  # At least 3 sections with headers
                "message": "Structured format with multiple sections"
            },
            "sufficient_length": {
                "passed": len(text) > 500,
                "message": f"Response length: {len(text)} chars"
            },
            "has_patent_references": {
                "passed": "US20" in text or "patent" in text.lower(),
                "message": "Patent references included"
            }
        }
        
        passed_count = sum(1 for check in checks.values() if check["passed"])
        total_count = len(checks)
        
        result = {
            "overall_passed": passed_count >= 4,  # At least 4/5 checks
            "score": f"{passed_count}/{total_count}",
            "checks": checks,
            "response_length": len(text)
        }
        
        logger.info("=" * 70)
        logger.info("📋 RESPONSE VALIDATION")
        logger.info("=" * 70)
        logger.info(f"Overall: {'✅ PASS' if result['overall_passed'] else '⚠️  PARTIAL PASS'}")
        logger.info(f"Score: {result['score']}")
        
        for check_name, check_result in checks.items():
            status = "✅" if check_result["passed"] else "❌"
            logger.info(f"  {status} {check_name}: {check_result['message']}")
        
        logger.info("=" * 70)
        
        return result
        
    except Exception as e:
        logger.error(f"❌ Validation failed: {e}", exc_info=True)
        return {
            "overall_passed": False,
            "error": str(e)
        }

# ==============================================================================
# SECTION 7: Execution with Full Observability
# ==============================================================================

logger.info("Initializing runner with custom observability plugin...")

# Create custom observability plugin
observability_plugin = ObservabilityPlugin(metrics)

# Initialize runner with both logging and custom observability
research_runner = InMemoryRunner(
    agent=PATENT_ROOT_AGENT,
    plugins=[
        LoggingPlugin(),           # ADK's standard logging
        observability_plugin       # Our custom agent tracking
    ]
)

logger.info("✅ Runner initialized with dual plugin system:")
logger.info("   - LoggingPlugin: Standard ADK logging")
logger.info("   - ObservabilityPlugin: Custom agent/metrics tracking")

# Define user query
# user_query = "Find white spots in the R&D area of advanced superconducting materials for energy."
user_query = "Find white spots in the R&D area of butterfly valves for gas and oil industries."
# user_query = "Find white spots in the R&D area of pulse combustors."

async def run_research():
    """Execute research pipeline with comprehensive error handling and metrics"""
    try:
        metrics.start_pipeline()
        
        logger.info(f"📝 User Query: {user_query}")
        
        response = await research_runner.run_debug(user_query, verbose=True)
        
        metrics.end_pipeline(success=True)
        
        # Log plugin summary
        plugin_summary = observability_plugin.get_summary()
        logger.info("=" * 70)
        logger.info("PLUGIN SUMMARY")
        logger.info(f"Total Agent Invocations: {plugin_summary['total_agent_invocations']}")
        logger.info(f"Total LLM Requests: {plugin_summary['total_llm_requests']}")
        logger.info(f"Completed Agents: {plugin_summary['completed_agents']}")
        logger.info("=" * 70)
        
        # Flush logs to ensure everything is written
        for handler in logging.getLogger().handlers:
            handler.flush()
        
        return response
        
    except Exception as e:
        metrics.end_pipeline(success=False)
        logger.error(f"❌ Pipeline execution failed: {type(e).__name__}: {e}", exc_info=True)
        
        # Flush logs even on error
        for handler in logging.getLogger().handlers:
            handler.flush()
        
        return {"error": str(e), "status": "failed"}

# ==============================================================================
# SECTION 8: Main Execution Block
# ==============================================================================

print("\n" + "=" * 70)
print("PATENT RESEARCH ASSISTANT MVP - ENHANCED OBSERVABILITY")
print("=" * 70)
print(f"Query: {user_query}")
print("=" * 70 + "\n")

try:
    response = await run_research()
    
    if isinstance(response, dict) and response.get("status") == "failed":
        logger.error("❌ Research failed")
        print(f"\n❌ ERROR: {response['error']}")
    else:
        logger.info("✅ Research completed successfully")
        
        # Validate response quality
        validation_result = validate_response(response)
        
        # Export metrics
        metrics_file = metrics.export_metrics()
        
        # Get plugin summary
        plugin_summary = observability_plugin.get_summary()
        
        print("\n" + "=" * 70)
        print("✅ SUCCESS - RESEARCH COMPLETED")
        print("=" * 70)
        print(f"Validation Score: {validation_result['score']}")
        print(f"Agent Invocations: {plugin_summary['total_agent_invocations']}")
        print(f"LLM Requests: {plugin_summary['total_llm_requests']}")
        print(f"Metrics Exported: {metrics_file}")
        print(f"Logs Saved: {LOG_FILENAME}")
        print("=" * 70)

except Exception as e:
    logger.critical(f"💥 Critical failure: {type(e).__name__}: {e}", exc_info=True)
    print(f"\n💥 CRITICAL FAILURE: {e}")

# ==============================================================================
# SECTION 9: Summary and Cleanup
# ==============================================================================

print("\n" + "=" * 70)
print("OBSERVABILITY SUMMARY")
print("=" * 70)
print(f"📊 Total Tool Calls: {len(metrics.tool_metrics)}")
print(f"🤖 Total Agent Invocations: {observability_plugin.agent_count}")
print(f"🔄 Completed Agents: {len([m for m in metrics.agent_metrics if m.end_time is not None])}")
print(f"🧠 LLM API Requests: {observability_plugin.llm_request_count}")
if metrics.pipeline_end_time and metrics.pipeline_start_time:
    print(f"⏱️  Total Duration: {metrics.pipeline_end_time - metrics.pipeline_start_time:.2f}s")
print(f"📄 Log File: {LOG_FILENAME}")
print("=" * 70)

logger.info("Patent Research Assistant execution complete")

# Final log flush to ensure everything is written
for handler in logging.getLogger().handlers:
    handler.flush()

2025-11-27 08:06:11 | INFO     | __main__                  | Initializing Patent Research Assistant MVP
2025-11-27 08:06:11 | INFO     | __main__                  | Log file: /kaggle/working/logs/patent_research_20251127_080611.log
2025-11-27 08:06:11 | INFO     | __main__                  | ✅ API credentials configured
2025-11-27 08:06:11 | INFO     | __main__                  | ✅ Model configured: gemini-2.5-flash-lite with retry logic
2025-11-27 08:06:11 | INFO     | __main__                  | ✅ Tools initialized with instrumentation
2025-11-27 08:06:11 | INFO     | __main__                  | Configuring agent hierarchy...
2025-11-27 08:06:11 | INFO     | __main__                  | ✅ Agent hierarchy configured
2025-11-27 08:06:11 | INFO     | __main__                  |    - Root: PatentRootAgent
2025-11-27 08:06:11 | INFO     | __main__                  |    - Pipeline: SequentialResearchPipeline
2025-11-27 08:06:11 | INFO     | __main__                  |    - Parallel: Patents